# Transformer English → Vietnamese

Notebook điều phối pipeline Transformer đã hiện đại hóa. Dữ liệu, checkpoint, log và kết quả đều dùng cấu trúc được mô tả trong spec.md. Notebook legacy ban đầu được giữ trong thư mục legacy để tham khảo.

## 1. Mount Google Drive và chọn chế độ

Sửa PROJECT_ROOT cho đúng thư mục dự án. RUN_MODE nhận dry-run, smoke hoặc full.

In [ ]:
from pathlib import Path
import os, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trong Google Colab; bỏ qua mount Drive.')

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP')
RUN_MODE = 'dry-run'  # 'dry-run', 'smoke', hoặc 'full'
DEVICE = 'auto'
assert RUN_MODE in {'dry-run', 'smoke', 'full'}
print('Project root:', PROJECT_ROOT)


## 2. Cài dependency

In [ ]:
requirements = PROJECT_ROOT / 'transformer_colab' / 'requirements-colab.txt'
if not requirements.is_file():
    raise FileNotFoundError(requirements)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)


Sau khi cell cài đặt thay đổi PyTorch, Colab có thể yêu cầu restart runtime. Khi đó mount lại Drive và chạy lại từ đầu.

## 3. Kiểm tra dữ liệu dùng chung

In [ ]:
os.chdir(PROJECT_ROOT)
validate_command = [
    sys.executable,
    'shared/scripts/validate_data.py',
    '--data-dir', 'shared/data/processed',
    '--manifest', 'shared/data/manifest.json',
]
subprocess.run(validate_command, check=True)


## 4. Huấn luyện hoặc smoke test

In [ ]:
train_command = [
    sys.executable,
    '-m', 'transformer_colab.src.train',
    '--config', 'transformer_colab/configs/baseline.json',
    '--device', DEVICE,
]
if RUN_MODE == 'dry-run':
    train_command.append('--dry-run')
elif RUN_MODE == 'smoke':
    train_command.append('--smoke')
print('Running:', ' '.join(train_command))
subprocess.run(train_command, check=True)


## 5. Dịch tập test

Cell này chỉ chạy khi đã có best.pt từ smoke/full training.

In [ ]:
checkpoint = PROJECT_ROOT / 'transformer_colab/checkpoints/best.pt'
if checkpoint.is_file():
    infer_command = [
        sys.executable,
        '-m', 'transformer_colab.src.infer',
        '--checkpoint', str(checkpoint),
        '--input', 'shared/data/processed/tst2013.en',
        '--output', 'transformer_colab/outputs/test.hyp.vi',
        '--beam-width', '3',
        '--device', DEVICE,
    ]
    subprocess.run(infer_command, check=True)
else:
    print('Chưa có best.pt; bỏ qua inference trong chế độ', RUN_MODE)


## 6. Đánh giá chuẩn hóa

In [ ]:
hypothesis = PROJECT_ROOT / 'transformer_colab/outputs/test.hyp.vi'
if hypothesis.is_file():
    evaluate_command = [
        sys.executable,
        'shared/evaluation/evaluate.py',
        '--reference', 'shared/data/processed/tst2013.vi',
        '--hypothesis', str(hypothesis),
        '--method', 'transformer',
        '--metadata', 'transformer_colab/outputs/run_metadata_infer.json',
        '--output', 'shared/evaluation/results/transformer.json',
    ]
    subprocess.run(evaluate_command, check=True)
else:
    print('Chưa có hypothesis; bỏ qua evaluation.')
